This file performs the following Steps. 

QC Filters the RAW Count matrices of GEO objects  
Normalizes then log transform per datasets  
Merge Geodatasets into two groups: Liver and Immune   
For each group  
Filter to genes expressed in > 10 cells per dataset  
Take intersection of genes across all datasets  
Concatentate on shared gene list (inner join)  
HVG selection with batch_key = "dataset_id", n_top = 5000  
Batch correction scVI?  

In [1]:
from pathlib import Path

import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse
from scipy.stats import median_abs_deviation
import numpy as np

# Set Base directory to the location of this script
BASE = Path('c:/Users/ankit/Documents/scFM/train_data/')

# Set working directory to the location of this script 
data_dir = BASE / 'h5s_common_directory_WIP_may_24'
# data_dir = BASE / 'h5s_common_directory_V2_WIP'
# data_dir = BASE / 'scGPT_data'

# Print contents of the data directory for verification
print(f"Contents of data directory ({data_dir}):")
for item in data_dir.iterdir():
    print(item.name)

Contents of data directory (c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_WIP_may_24):
GSE159977.h5ad
GSE174748.h5ad
GSE185477.h5ad
GSE189600.h5ad
GSE190487.h5ad
GSE192740.h5ad
GSE202379.h5ad
GSE212837.h5ad
GSE270488.h5ad


In [2]:
# Read in each h5ad file with the Anndata object variable name being it's file name without the extension and store in dictionary
adata_dict = {}

for h5ad_file in data_dir.glob('*.h5ad'):
    adata_name = h5ad_file.stem  # Get file name without extension
    adata_dict[adata_name] = ad.read_h5ad(h5ad_file)  # Read h5ad file and store in dictionary



c:\Users\ankit\miniconda3\envs\scgpt\Lib\site-packages\anndata\_core\anndata.py:1878: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Pre Filtering Cell counts

In [3]:
for adata_name, adata in adata_dict.items():
    print(f"\nProcessing Anndata object: {adata_name}")
    print(f"Shape of the Anndata object: {adata.shape}")
    print(f"Number of genes: {adata.n_vars}")
    print(f"Number of cells: {adata.n_obs}")


Processing Anndata object: GSE159977
Shape of the Anndata object: (10923732, 33694)
Number of genes: 33694
Number of cells: 10923732

Processing Anndata object: GSE174748
Shape of the Anndata object: (4524728, 33538)
Number of genes: 33538
Number of cells: 4524728

Processing Anndata object: GSE185477
Shape of the Anndata object: (9560531, 45068)
Number of genes: 45068
Number of cells: 9560531

Processing Anndata object: GSE189600
Shape of the Anndata object: (65879, 36601)
Number of genes: 36601
Number of cells: 65879

Processing Anndata object: GSE190487
Shape of the Anndata object: (810280, 33538)
Number of genes: 33538
Number of cells: 810280

Processing Anndata object: GSE192740
Shape of the Anndata object: (102004, 32738)
Number of genes: 32738
Number of cells: 102004

Processing Anndata object: GSE202379
Shape of the Anndata object: (68319, 31257)
Number of genes: 31257
Number of cells: 68319

Processing Anndata object: GSE212837
Shape of the Anndata object: (252669, 36601)
Num

In [4]:
for name, adata in adata_dict.items():
    sc.pp.filter_cells(adata, min_genes=100)
    sc.pp.filter_genes(adata, min_cells=3)

c:\Users\ankit\miniconda3\envs\scgpt\Lib\site-packages\anndata\_core\anndata.py:1878: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
c:\Users\ankit\miniconda3\envs\scgpt\Lib\site-packages\anndata\_core\anndata.py:1878: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
c:\Users\ankit\miniconda3\envs\scgpt\Lib\site-packages\anndata\_core\anndata.py:1878: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Post Filtering Cell Counts

In [5]:

for adata_name, adata in adata_dict.items():
    print(f"\nProcessing Anndata object: {adata_name}")
    print(f"Shape of the Anndata object: {adata.shape}")
    print(f"Number of genes: {adata.n_vars}")
    print(f"Number of cells: {adata.n_obs}")


Processing Anndata object: GSE159977
Shape of the Anndata object: (115510, 23993)
Number of genes: 23993
Number of cells: 115510

Processing Anndata object: GSE174748
Shape of the Anndata object: (199524, 27934)
Number of genes: 27934
Number of cells: 199524

Processing Anndata object: GSE185477
Shape of the Anndata object: (493369, 37469)
Number of genes: 37469
Number of cells: 493369

Processing Anndata object: GSE189600
Shape of the Anndata object: (65879, 32277)
Number of genes: 32277
Number of cells: 65879

Processing Anndata object: GSE190487
Shape of the Anndata object: (51223, 19059)
Number of genes: 19059
Number of cells: 51223

Processing Anndata object: GSE192740
Shape of the Anndata object: (101378, 30704)
Number of genes: 30704
Number of cells: 101378

Processing Anndata object: GSE202379
Shape of the Anndata object: (68319, 30596)
Number of genes: 30596
Number of cells: 68319

Processing Anndata object: GSE212837
Shape of the Anndata object: (252664, 33557)
Number of gen

In [6]:
# Update n_counts, n_genes
for name, adata in adata_dict.items():
    adata.obs["n_counts"] = adata.X.sum(axis=1).A1 if sparse.issparse(adata.X) else adata.X.sum(axis=1)
    print(f"{name} did not have an n_counts column, so one was created by summing the adata.X matrix")
    
    adata.obs["n_genes"] = (adata.X > 0).sum(axis=1).A1 if sparse.issparse(adata.X) else (adata.X > 0).sum(axis=1)
    print(f"{name} did not have an n_genes column, so one was created by summing the adata.X matrix")

GSE159977 did not have an n_counts column, so one was created by summing the adata.X matrix
GSE159977 did not have an n_genes column, so one was created by summing the adata.X matrix
GSE174748 did not have an n_counts column, so one was created by summing the adata.X matrix
GSE174748 did not have an n_genes column, so one was created by summing the adata.X matrix
GSE185477 did not have an n_counts column, so one was created by summing the adata.X matrix
GSE185477 did not have an n_genes column, so one was created by summing the adata.X matrix
GSE189600 did not have an n_counts column, so one was created by summing the adata.X matrix
GSE189600 did not have an n_genes column, so one was created by summing the adata.X matrix
GSE190487 did not have an n_counts column, so one was created by summing the adata.X matrix
GSE190487 did not have an n_genes column, so one was created by summing the adata.X matrix
GSE192740 did not have an n_counts column, so one was created by summing the adata.X 

QC Annotation

In [7]:
for name, adata in adata_dict.items():
    # Annotate genes for QC metrics.
    adata.var["mt"] = adata.var_names.str.startswith("MT-")
    adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
    adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

    # Calculate QC metrics.
    sc.pp.calculate_qc_metrics(
        adata,
        qc_vars=["mt", "ribo", "hb"],
        inplace=True,
        percent_top=[20],
        log1p=True,
    )



In [8]:
# Pre filtering Mito %
for name, adata in adata_dict.items():
    print(adata.shape)

for name, adata in adata_dict.items():
    adata = adata[adata.obs["pct_counts_mt"] < 10, :]
    adata = adata[adata.obs["n_counts"] <= (np.median(adata.obs["n_counts"]) + 5 * median_abs_deviation(adata.obs["n_counts"])), :]
    adata_dict[name] = adata

print("Post filtering mito %)")
for name, adata in adata_dict.items():
    print(adata.shape)

(115510, 23993)
(199524, 27934)
(493369, 37469)
(65879, 32277)
(51223, 19059)
(101378, 30704)
(68319, 30596)
(252664, 33557)
(18384, 21251)
Post filtering mito %)
(106851, 23993)
(19038, 27934)
(125790, 37469)
(55074, 32277)
(41995, 19059)
(69758, 30704)
(64329, 30596)
(182302, 33557)
(18376, 21251)


In [9]:
# Save each anndata object in the dictionary to a new h5ad file with the same GSE file name in the directory called 'h5s_common_directory_V2_WIP'

output_dir = BASE / 'h5s_common_directory_V2_WIP'
output_dir.mkdir(exist_ok=True)  # Create output directory if it doesn't exist
for name, adata in adata_dict.items():
    output_file = output_dir / f"{name}.h5ad"
    adata.write_h5ad(output_file)
    print(f"Saved {name} to {output_file}")

Saved GSE159977 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIP\GSE159977.h5ad
Saved GSE174748 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIP\GSE174748.h5ad
Saved GSE185477 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIP\GSE185477.h5ad
Saved GSE189600 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIP\GSE189600.h5ad
Saved GSE190487 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIP\GSE190487.h5ad
Saved GSE192740 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIP\GSE192740.h5ad
Saved GSE202379 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIP\GSE202379.h5ad
Saved GSE212837 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIP\GSE212837.h5ad
Saved GSE270488 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIP\GSE270488.h5ad


Normalization Sanity Check

In [10]:
import numpy as np
from scipy import sparse

def row_sums(M):
    return np.asarray(M.sum(axis=1)).ravel()

def expm1_row_sums(M):
    if sparse.issparse(M):
        M2 = M.copy()
        M2.data = np.expm1(M2.data)
        return np.asarray(M2.sum(axis=1)).ravel()
    else:
        return np.expm1(M).sum(axis=1)

def matrix_summary(M, name):
    data = M.data if sparse.issparse(M) else np.asarray(M).ravel()
    data = data[np.isfinite(data)]

    print(f"\n{name}")
    print("-" * len(name))
    print("shape:", M.shape)
    print("sparse:", sparse.issparse(M))
    print("min:", data.min())
    print("max:", data.max())
    print("mean nonzero/value:", data.mean())
    print("integer-like values:", np.mean(np.isclose(data, np.round(data))) > 0.99)

    rs = row_sums(M)
    print("row sum summary:")
    print(np.percentile(rs, [0, 1, 25, 50, 75, 99, 100]))

    ers = expm1_row_sums(M)
    print("expm1(row) sum summary:")
    print(np.percentile(ers, [0, 1, 25, 50, 75, 99, 100]))

    print("close to 10,000 after expm1?:", np.mean(np.isclose(ers, 10000, rtol=1e-3, atol=1)))



In [11]:

for name, adata in adata_dict.items():
    print("============================")
    print(f"Looking at: {name}\n")
    matrix_summary(adata.X, "adata.X")
    if "counts" in adata.layers:
        matrix_summary(adata.layers["counts"], "adata.layers['counts']")

    if adata.raw is not None:
        matrix_summary(adata.raw.X, "adata.raw.X")

Looking at: GSE159977


adata.X
-------
shape: (106851, 23993)
sparse: True
min: 1.0
max: 10295.0
mean nonzero/value: 2.8789535
integer-like values: True
row sum summary:
[  106.    131.    882.5  3583.   5523.  12951.5 15198. ]


C:\Users\ankit\AppData\Local\Temp\ipykernel_26884\432754156.py:10: RuntimeWarning: overflow encountered in expm1
  M2.data = np.expm1(M2.data)
c:\Users\ankit\miniconda3\envs\scgpt\Lib\site-packages\scipy\sparse\_compressed.py:534: RuntimeWarning: overflow encountered in reduceat
  value = ufunc.reduceat(data,
c:\Users\ankit\miniconda3\envs\scgpt\Lib\site-packages\numpy\lib\_function_base_impl.py:4596: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a


expm1(row) sum summary:
[1.99102646e+02 4.39938065e+02 2.29378386e+27            nan
            nan            nan            nan]
close to 10,000 after expm1?: 3.7435307109900705e-05

adata.layers['counts']
----------------------
shape: (106851, 23993)
sparse: True
min: 1.0
max: 10295.0
mean nonzero/value: 2.8789535
integer-like values: True
row sum summary:
[  106.    131.    882.5  3583.   5523.  12951.5 15198. ]
expm1(row) sum summary:
[1.99102646e+02 4.39938065e+02 2.29378386e+27            nan
            nan            nan            nan]
close to 10,000 after expm1?: 3.7435307109900705e-05
Looking at: GSE174748


adata.X
-------
shape: (19038, 27934)
sparse: True
min: 1
max: 986
mean nonzero/value: 1.851752430869053
integer-like values: True
row sum summary:
[ 102.    113.    255.    777.   2550.75 6834.63 7140.  ]
expm1(row) sum summary:
[1.78217239e+002 2.33721004e+002 9.61285105e+003 1.44625707e+012
 1.11286375e+036 1.74395225e+146             nan]
close to 10,000 after exp

MemoryError: Unable to allocate 2.68 GiB for an array with shape (359056670,) and data type float64

The following updates adata.X to be normalized log counts

In [12]:
import scanpy as sc
import numpy as np
from scipy import sparse

to_fix = [
    "GSE159977",
    "GSE174748",
    "GSE185477",
    "GSE189600",
    "GSE190487",
    "GSE192740",
    "GSE202379",
    "GSE212837",
]

def is_integer_like_matrix(M, sample_n=100000):
    """Quick check that matrix values look like raw counts."""
    data = M.data if sparse.issparse(M) else np.asarray(M).ravel()
    if data.size == 0:
        return False

    if data.size > sample_n:
        idx = np.random.choice(data.size, sample_n, replace=False)
        data = data[idx]

    return np.mean(np.isclose(data, np.round(data))) > 0.99


for name, adata in adata_dict.items():
    print("============================")
    print(f"Processing: {name}")

    if name not in to_fix:
        print(f"Skipping {name}: appears already log1p-normalized to 10,000.")
        continue

    # Decide source of raw counts
    # For these objects, your output suggests adata.layers['counts'] and adata.X are both raw counts.
    if "counts" in adata.layers and is_integer_like_matrix(adata.layers["counts"]):
        counts_source = adata.layers["counts"].copy()
        print("Using existing adata.layers['counts'] as raw count source.")
    elif is_integer_like_matrix(adata.X):
        counts_source = adata.X.copy()
        print("Using adata.X as raw count source.")
    else:
        raise ValueError(
            f"{name}: Could not confidently identify raw counts. "
            "Check .X and .layers manually before normalizing."
        )

    # Preserve raw counts before modification
    # This is the important safe backup.
    adata.layers["counts"] = counts_source.copy()
    adata.layers["raw_counts"] = counts_source.copy()

    # Set X to raw counts before normalization
    adata.X = counts_source.copy()

    # Normalize to 10,000 counts per cell
    sc.pp.normalize_total(adata, target_sum=1e4)

    # Natural log1p transform
    sc.pp.log1p(adata)

    print(f"Done: {name}")
    print("Raw counts saved in:")
    print("  adata.layers['counts']")
    print("  adata.layers['raw_counts']")
    print("  adata.raw.X")

Processing: GSE159977
Using existing adata.layers['counts'] as raw count source.


C:\Users\ankit\AppData\Local\Temp\ipykernel_26884\2373997060.py:53: ImplicitModificationWarning: Setting element `.layers['counts']` of view, initializing view as actual.
  adata.layers["counts"] = counts_source.copy()
c:\Users\ankit\miniconda3\envs\scgpt\Lib\site-packages\anndata\_core\anndata.py:1878: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Done: GSE159977
Raw counts saved in:
  adata.layers['counts']
  adata.layers['raw_counts']
  adata.raw.X
Processing: GSE174748
Using existing adata.layers['counts'] as raw count source.


C:\Users\ankit\AppData\Local\Temp\ipykernel_26884\2373997060.py:53: ImplicitModificationWarning: Setting element `.layers['counts']` of view, initializing view as actual.
  adata.layers["counts"] = counts_source.copy()


Done: GSE174748
Raw counts saved in:
  adata.layers['counts']
  adata.layers['raw_counts']
  adata.raw.X
Processing: GSE185477
Using existing adata.layers['counts'] as raw count source.
Done: GSE185477
Raw counts saved in:
  adata.layers['counts']
  adata.layers['raw_counts']
  adata.raw.X
Processing: GSE189600
Using existing adata.layers['counts'] as raw count source.
Done: GSE189600
Raw counts saved in:
  adata.layers['counts']
  adata.layers['raw_counts']
  adata.raw.X
Processing: GSE190487
Using existing adata.layers['counts'] as raw count source.
Done: GSE190487
Raw counts saved in:
  adata.layers['counts']
  adata.layers['raw_counts']
  adata.raw.X
Processing: GSE192740
Using existing adata.layers['counts'] as raw count source.
Done: GSE192740
Raw counts saved in:
  adata.layers['counts']
  adata.layers['raw_counts']
  adata.raw.X
Processing: GSE202379
Using existing adata.layers['counts'] as raw count source.
Done: GSE202379
Raw counts saved in:
  adata.layers['counts']
  adata.

Save the files again

In [13]:
# Save each anndata object in the dictionary to a new h5ad file with the same GSE file name in the directory called 'h5s_common_directory_V2_WIP'

output_dir = BASE / 'h5s_common_directory_V2_WIP'
output_dir.mkdir(exist_ok=True)  # Create output directory if it doesn't exist
for name, adata in adata_dict.items():
    output_file = output_dir / f"{name}.h5ad"
    adata.write_h5ad(output_file)
    print(f"Saved {name} to {output_file}")

Saved GSE159977 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIP\GSE159977.h5ad
Saved GSE174748 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIP\GSE174748.h5ad
Saved GSE185477 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIP\GSE185477.h5ad
Saved GSE189600 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIP\GSE189600.h5ad
Saved GSE190487 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIP\GSE190487.h5ad
Saved GSE192740 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIP\GSE192740.h5ad
Saved GSE202379 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIP\GSE202379.h5ad
Saved GSE212837 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIP\GSE212837.h5ad
Saved GSE270488 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIP\GSE270488.h5ad
